# GROUP 10: University of Vienna – Business Intelligence 1

In [45]:
import pandas as pd
import numpy as np

In [3]:
income_data = pd.read_csv("../data/raw/vie-bez-biz-ecn-inc-sex-2002f.csv", sep=";", skiprows=1)
unemployment_data = pd.read_csv("../data/raw/vie-bez-biz-emp-sex-uep-2002f.csv", sep=";", skiprows=1)

In [4]:
income_data.head()

,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,INC_TOT_VALUE,INC_MAL_VALUE,INC_FEM_VALUE,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,AT13,90000,90000,2002,20021231,18.217,20.709,15.424,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AT13,90100,90100,2002,20021231,25.463,31.961,18.536,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AT13,90200,90200,2002,20021231,16.439,18.301,14.282,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AT13,90300,90300,2002,20021231,18.701,21.444,15.804,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AT13,90400,90400,2002,20021231,20.325,23.641,16.876,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
unemployment_data.head()

,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,SEX,UEP_VALUE,UEP_DENSITY,Unnamed: 8,Unnamed: 9
0,AT13,90000,90000,2002,2002,0,74.894,"67,83",NaN,NaN
1,AT13,90100,90100,2002,2002,0,433.000,"35,32",NaN,NaN
2,AT13,90200,90200,2002,2002,0,4.784,"76,98",NaN,NaN
3,AT13,90300,90300,2002,2002,0,3.957,"68,25",NaN,NaN
4,AT13,90400,90400,2002,2002,0,1.140,"55,68",NaN,NaN


## Data Transformation and Integration

### Dim_Sex

In [28]:
dim_sex_data = {
    'id':["0", "1"],
    'code':["m", "f"],
    'name':["Male", "Female"],
}
dim_sex_df = pd.DataFrame(dim_sex_data)
dim_sex_df.to_csv('../data/created/dim_sex_data.csv', sep=';', index=False) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html#pandas.DataFrame.to_csv

### Dim_Time

In [21]:
#Check if both REF_YEAR and REF_DATE have the same values for unemployment_data
unemployment_data["REF_YEAR"].equals(unemployment_data["REF_DATE"]) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.equals.html

True

In [22]:
#Check if both REF_YEAR and REF_DATE have the same year for income_data
income_data["REF_YEAR"].astype(str).equals(income_data["REF_DATE"].astype(str).str[:4]) #https://stackoverflow.com/questions/36505847/substring-of-an-entire-column-in-pandas-dataframe

True

In [14]:
year_min = min(income_data["REF_YEAR"].min(), unemployment_data["REF_YEAR"].min())
year_min

2002

In [15]:
year_max = max(income_data["REF_YEAR"].max(), unemployment_data["REF_YEAR"].max())
year_max

2023

In [33]:
dim_date_data = {
    'id':list(range(0, year_max-year_min+1,1)), #https://stackoverflow.com/questions/40706034/how-to-create-a-list-of-a-range-with-incremental-step
    'year':list(range(year_min,year_max+1,1)) #https://www.w3schools.com/python/ref_func_range.asp
}
dim_date_df = pd.DataFrame(dim_date_data) 
dim_date_df.to_csv('../data/created/dim_date_data.csv', sep=';', index=False) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html#pandas.DataFrame.to_csv

### Dim_District

In [25]:
#Check if both DISTRICT_CODE and SUB_DISTRICT_CODE have the same values for income_data
income_data["DISTRICT_CODE"].equals(income_data["SUB_DISTRICT_CODE"]) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.equals.html

True

In [26]:
#Check if both DISTRICT_CODE and SUB_DISTRICT_CODE have the same values for income_data
unemployment_data["DISTRICT_CODE"].equals(unemployment_data["SUB_DISTRICT_CODE"]) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.equals.html

True

In [38]:
#Check how many states
income_data["NUTS"].unique()

array(['AT13'], dtype=object)

In [47]:
#Check how many districts
districts = income_data["DISTRICT_CODE"].astype(str).str[1:3].unique() #https://numpy.org/doc/stable/reference/generated/numpy.ndarray.tolist.html
districts = list(districts)
#remove district 00, since I assume it is the total)
districts.pop(0) #https://www.geeksforgeeks.org/python/python-removing-first-element-of-list/

'00'

In [65]:
dim_district_data = {
    'id':list(range(0, len(districts), 1)), #https://stackoverflow.com/questions/40706034/how-to-create-a-list-of-a-range-with-incremental-step
    'state':[income_data["NUTS"].unique()[0]] * len(districts), #https://stackoverflow.com/questions/3459098/create-list-of-single-item-repeated-n-times
    'district':districts,
    'area_km2':[""] * len(districts)
}
dim_district_df = pd.DataFrame(dim_district_data) 
dim_district_df.to_csv('../data/created/dim_district_data.csv', sep=';', index=False) #https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html#pandas.DataFrame.to_csv

## Data Cleaning